# Base models in plain text: is the outvoted map a product of instruction tuning? (preregistered: `docs/preregistration_base_plain.md`)

The flow set's 2,400 "which is farther east / north, C or R?" questions, asked to **pretrained-only (base) models** in a
plain-text prompt ending in `Answer:`, and to the instruct models in the same plain prompt (format control). Qwen3-8B,
Llama 3.1 8B, Gemma 3 12B (8-bit weights on an L4).

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. On Hugging Face, while logged in, open **https://huggingface.co/google/gemma-3-12b-pt** and
   **https://huggingface.co/meta-llama/Llama-3.1-8B** and accept the licence if asked (the base models are gated
   separately from the instruct ones). Cell 4 checks access and says which are missing.
3. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
4. Runtime → Change runtime type → **L4** (or A100 / H100). Then **Run all**. About 1.5–2 hours on an L4, mostly
   downloading six models.

Results are saved to Drive as they are produced; after a disconnect, just press Run all again (finished runs are
skipped). The last cell downloads `base_plain_results.zip`; send that file back.

In [ ]:
# 1. GPU check
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 / A100 / H100"
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
assert GB > 20, f"Only {GB:.0f} GB of GPU memory: pick an L4, A100 or H100"
BIG = GB > 35
GEMMA_PT = "gemma3-12b-pt" if BIG else "gemma3-12b-pt-8bit"   # preregistered: bf16 on >= 35 GB, else 8-bit weights
GEMMA_IT = "gemma3-12b" if BIG else "gemma3-12b-8bit"
BS_8B, BS_GEMMA = (32, 32) if BIG else (16, 8)        # batch sizes: memory only
print(f"{GB:.0f} GB: Llama and Qwen in bf16; Gemma 3 12B as {GEMMA_PT} / {GEMMA_IT}")

In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert os.path.exists("/content/geomap/scripts/base_plain.py"), "Old bundle: upload the new geomap_colab.zip"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
os.makedirs(f"{RES}/base_plain", exist_ok=True)
print("already saved:", sorted(os.listdir(f"{RES}/base_plain")))

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' imports); bitsandbytes for 8-bit Gemma
!pip -q install "transformers==4.57.3" nnsight accelerate bitsandbytes
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Base/plain] Colab run started (or resumed)")
# Which gated models can this token open? (the base models need their own licence acceptance)
from huggingface_hub import hf_hub_download
for repo in ["Qwen/Qwen3-8B-Base", "meta-llama/Llama-3.1-8B", "google/gemma-3-12b-pt", "Qwen/Qwen3-8B",
             "meta-llama/Llama-3.1-8B-Instruct", "google/gemma-3-12b-it"]:
    try:
        hf_hub_download(repo, "config.json", token=os.environ["HF_TOKEN"])
        print("ok     ", repo)
    except Exception as e:
        print("NO ACCESS", repo, "-> accept the licence on its Hugging Face page, then rerun this cell", type(e).__name__)


In [ ]:
# 5. One run per model; each model's download is deleted afterwards (six models would fill the disk)
from geomap.models import MODELS
def run(key, bs):
    cmd = (f"cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 "
           f"python scripts/base_plain.py --model {key} --fmt plain --batch-size {bs} 2>&1 | grep -v -i warning")
    get_ipython().system(cmd)
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--" + MODELS[key].replace("/", "--")), ignore_errors=True)
    print("done:", sorted(f for f in os.listdir(f"{RES}/base_plain") if f.endswith(".done")))

In [ ]:
# 6. Qwen3-8B-Base
run("qwen3-8b-base", BS_8B)

In [ ]:
# 7. Llama 3.1 8B (base)
run("llama3.1-8b-base", BS_8B)

In [ ]:
# 8. Gemma 3 12B pt (base; longest)
run(GEMMA_PT, BS_GEMMA)

In [ ]:
# 9. Format control: Qwen3-8B (instruct) in the plain prompt
run("qwen3-8b", BS_8B)

In [ ]:
# 10. Format control: Llama 3.1 8B Instruct in the plain prompt
run("llama3.1-8b", BS_8B)

In [ ]:
# 11. Format control: Gemma 3 12B it in the plain prompt
run(GEMMA_IT, BS_GEMMA)

In [ ]:
# 12. Overall verdict (base models)
!cd /content/geomap && PYTHONPATH=/content/geomap/src python scripts/base_plain.py --summary 2>&1 | grep -v -i warning

In [ ]:
# 13. One zip of the results (without the cached residuals) + the results log, on Drive and downloaded here
import glob
with zipfile.ZipFile("/content/base_plain_results.zip", "w") as z:
    for f in glob.glob(f"{RES}/base_plain/*"):
        if not f.endswith(".pt"):
            z.write(f, "results/base_plain/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
    print("\n".join(z.namelist()))
shutil.copy("/content/base_plain_results.zip", f"{D}/base_plain_results.zip")
notify("[Base/plain] zip saved: MyDrive/geomap/base_plain_results.zip")
from google.colab import files
files.download("/content/base_plain_results.zip")